All code was run on Google Colab (T4 GPU) using Jupyter notebooks. Total runtime is roughly 90 minutes. Epoch count was not tuned. No parameter sensitivity analysis was carried out.

In [1]:
!pip install tensorboard -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device}")

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False, num_workers=2)

cuda


In [2]:
model_alex = models.alexnet(weights=None).to(device)
model_alex.classifier[6] = nn.Linear(4096, 10) ## add new layer.
model_alex = model_alex.to(device)

optimizer = optim.Adam(model_alex.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/AlexNet_scratch")

In [3]:
for epoch in range(30):
    model_alex.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_alex(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/30 - Loss: {avg_loss:.4f}")
writer.close()

model_alex.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_alex(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"AlexNet Scratch Accuracy: {acc:.2f}%")
torch.save(model_alex.state_dict(), "alexnet_scratch.pth")
results = {"AlexNet_scratch": acc}

Epoch 1/30 - Loss: 1.5306
Epoch 2/30 - Loss: 1.0798
Epoch 3/30 - Loss: 0.8478
Epoch 4/30 - Loss: 0.6966
Epoch 5/30 - Loss: 0.5963
Epoch 6/30 - Loss: 0.5108
Epoch 7/30 - Loss: 0.4338
Epoch 8/30 - Loss: 0.3740
Epoch 9/30 - Loss: 0.3146
Epoch 10/30 - Loss: 0.2602
Epoch 11/30 - Loss: 0.2172
Epoch 12/30 - Loss: 0.1882
Epoch 13/30 - Loss: 0.1636
Epoch 14/30 - Loss: 0.1382
Epoch 15/30 - Loss: 0.1224
Epoch 16/30 - Loss: 0.1103
Epoch 17/30 - Loss: 0.1051
Epoch 18/30 - Loss: 0.0926
Epoch 19/30 - Loss: 0.0882
Epoch 20/30 - Loss: 0.0786
Epoch 21/30 - Loss: 0.0791
Epoch 22/30 - Loss: 0.0729
Epoch 23/30 - Loss: 0.0709
Epoch 24/30 - Loss: 0.0603
Epoch 25/30 - Loss: 0.0599
Epoch 26/30 - Loss: 0.0608
Epoch 27/30 - Loss: 0.0572
Epoch 28/30 - Loss: 0.0597
Epoch 29/30 - Loss: 0.0578
Epoch 30/30 - Loss: 0.0485
AlexNet Scratch Accuracy: 83.93%


In [4]:
model_alex2 = models.alexnet(weights=models.AlexNet_Weights.DEFAULT).to(device)
model_alex2.classifier[6] = nn.Linear(4096, 10)
model_alex2 = model_alex2.to(device)

optimizer = optim.Adam(model_alex2.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/AlexNet_pretrained")

for epoch in range(30):
    model_alex2.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_alex2(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/30 - Loss: {avg_loss:.4f}")
writer.close()

model_alex2.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_alex2(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"AlexNet Pretrained Accuracy: {acc:.2f}%")
torch.save(model_alex2.state_dict(), "alexnet_pretrained.pth")
results["AlexNet_pretrained"] = acc

Epoch 1/30 - Loss: 0.5752
Epoch 2/30 - Loss: 0.3100
Epoch 3/30 - Loss: 0.2106
Epoch 4/30 - Loss: 0.1515
Epoch 5/30 - Loss: 0.1142
Epoch 6/30 - Loss: 0.0906
Epoch 7/30 - Loss: 0.0740
Epoch 8/30 - Loss: 0.0639
Epoch 9/30 - Loss: 0.0566
Epoch 10/30 - Loss: 0.0577
Epoch 11/30 - Loss: 0.0476
Epoch 12/30 - Loss: 0.0454
Epoch 13/30 - Loss: 0.0423
Epoch 14/30 - Loss: 0.0417
Epoch 15/30 - Loss: 0.0381
Epoch 16/30 - Loss: 0.0357
Epoch 17/30 - Loss: 0.0348
Epoch 18/30 - Loss: 0.0288
Epoch 19/30 - Loss: 0.0381
Epoch 20/30 - Loss: 0.0295
Epoch 21/30 - Loss: 0.0313
Epoch 22/30 - Loss: 0.0272
Epoch 23/30 - Loss: 0.0292
Epoch 24/30 - Loss: 0.0301
Epoch 25/30 - Loss: 0.0260
Epoch 26/30 - Loss: 0.0275
Epoch 27/30 - Loss: 0.0262
Epoch 28/30 - Loss: 0.0238
Epoch 29/30 - Loss: 0.0263
Epoch 30/30 - Loss: 0.0203
AlexNet Pretrained Accuracy: 91.32%


In [5]:
import pandas as pd
pd.DataFrame(results.items(), columns=["Run", "Test Accuracy"]).to_csv("results.csv", index=False)
from google.colab import files
files.download("results.csv")
files.download("alexnet_scratch.pth")
files.download("alexnet_pretrained.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
transform_mnist = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset_mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)
testset_mnist  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

trainloader_mnist = torch.utils.data.DataLoader(trainset_mnist, batch_size=64, shuffle=True, num_workers=2)
testloader_mnist  = torch.utils.data.DataLoader(testset_mnist,  batch_size=64, shuffle=False, num_workers=2)
##Lab 0.2.2, using tanh but adopting it for grey scale
model_mnist = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 256), nn.Tanh(),
    nn.Linear(256, 10)
).to(device)

optimizer = optim.Adam(model_mnist.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/MNIST_CNN")

for epoch in range(30):
    model_mnist.train()
    running_loss = 0.0
    for inputs, labels in trainloader_mnist:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_mnist(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader_mnist)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/30 - Loss: {avg_loss:.4f}")
writer.close()

model_mnist.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader_mnist:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_mnist(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"MNIST Accuracy: {acc:.2f}%")
torch.save(model_mnist.state_dict(), "mnist_cnn.pth")
results["MNIST_CNN"] = acc

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.64MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.89MB/s]


Epoch 1/30 - Loss: 0.3595
Epoch 2/30 - Loss: 0.1096
Epoch 3/30 - Loss: 0.0707
Epoch 4/30 - Loss: 0.0525
Epoch 5/30 - Loss: 0.0425
Epoch 6/30 - Loss: 0.0345
Epoch 7/30 - Loss: 0.0298
Epoch 8/30 - Loss: 0.0243
Epoch 9/30 - Loss: 0.0213
Epoch 10/30 - Loss: 0.0174
Epoch 11/30 - Loss: 0.0144
Epoch 12/30 - Loss: 0.0129
Epoch 13/30 - Loss: 0.0103
Epoch 14/30 - Loss: 0.0090
Epoch 15/30 - Loss: 0.0077
Epoch 16/30 - Loss: 0.0061
Epoch 17/30 - Loss: 0.0051
Epoch 18/30 - Loss: 0.0047
Epoch 19/30 - Loss: 0.0037
Epoch 20/30 - Loss: 0.0033
Epoch 21/30 - Loss: 0.0025
Epoch 22/30 - Loss: 0.0037
Epoch 23/30 - Loss: 0.0015
Epoch 24/30 - Loss: 0.0016
Epoch 25/30 - Loss: 0.0019
Epoch 26/30 - Loss: 0.0012
Epoch 27/30 - Loss: 0.0021
Epoch 28/30 - Loss: 0.0006
Epoch 29/30 - Loss: 0.0015
Epoch 30/30 - Loss: 0.0004
MNIST Accuracy: 99.09%


In [7]:
transform_svhn = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset_svhn = torchvision.datasets.SVHN(root='./data', split='train', download=True, transform=transform_svhn)
testset_svhn  = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform_svhn)

trainloader_svhn = torch.utils.data.DataLoader(trainset_svhn, batch_size=64, shuffle=True, num_workers=2)
testloader_svhn  = torch.utils.data.DataLoader(testset_svhn,  batch_size=64, shuffle=False, num_workers=2)

100%|██████████| 182M/182M [00:26<00:00, 6.75MB/s]
100%|██████████| 64.3M/64.3M [00:17<00:00, 3.62MB/s]


In [8]:
optimizer = optim.Adam(model_mnist.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/SVHN_transfer")

# using MNIST pretrained weights as starting point for SVHN (transfer learning)
for epoch in range(30):
    model_mnist.train()
    running_loss = 0.0
    for inputs, labels in trainloader_svhn:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_mnist(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader_svhn)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/30 - Loss: {avg_loss:.4f}")
writer.close()

model_mnist.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader_svhn:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_mnist(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"SVHN Transfer Accuracy: {acc:.2f}%")
torch.save(model_mnist.state_dict(), "svhn_transfer.pth")
results["SVHN_transfer"] = acc

Epoch 1/30 - Loss: 1.1195
Epoch 2/30 - Loss: 0.6078
Epoch 3/30 - Loss: 0.4955
Epoch 4/30 - Loss: 0.4314
Epoch 5/30 - Loss: 0.3866
Epoch 6/30 - Loss: 0.3514
Epoch 7/30 - Loss: 0.3207
Epoch 8/30 - Loss: 0.2957
Epoch 9/30 - Loss: 0.2735
Epoch 10/30 - Loss: 0.2532
Epoch 11/30 - Loss: 0.2354
Epoch 12/30 - Loss: 0.2185
Epoch 13/30 - Loss: 0.2033
Epoch 14/30 - Loss: 0.1897
Epoch 15/30 - Loss: 0.1763
Epoch 16/30 - Loss: 0.1633
Epoch 17/30 - Loss: 0.1517
Epoch 18/30 - Loss: 0.1409
Epoch 19/30 - Loss: 0.1303
Epoch 20/30 - Loss: 0.1197
Epoch 21/30 - Loss: 0.1104
Epoch 22/30 - Loss: 0.1020
Epoch 23/30 - Loss: 0.0932
Epoch 24/30 - Loss: 0.0863
Epoch 25/30 - Loss: 0.0784
Epoch 26/30 - Loss: 0.0718
Epoch 27/30 - Loss: 0.0654
Epoch 28/30 - Loss: 0.0590
Epoch 29/30 - Loss: 0.0536
Epoch 30/30 - Loss: 0.0489
SVHN Transfer Accuracy: 88.08%


In [9]:
## Just forgot step 2 before step 3, so re-loading pre-fine tuning weights.
model_mnist.load_state_dict(torch.load("mnist_cnn.pth"))

model_mnist.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader_svhn:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_mnist(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc_direct = 100 * correct / total
print(f"MNIST model directly on SVHN (no retraining): {acc_direct:.2f}%")
results["MNIST_direct_SVHN"] = acc_direct

MNIST model directly on SVHN (no retraining): 22.88%


In [10]:
import pandas as pd
pd.DataFrame(results.items(), columns=["Run", "Test Accuracy"]).to_csv("results_task02.csv", index=False)

import shutil
shutil.make_archive("task02_runs", "zip", "runs")

from google.colab import files
files.download("results_task02.csv")
files.download("mnist_cnn.pth")
files.download("svhn_transfer.pth")
files.download("task02_runs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>